In [3]:
import numpy as np
import pandas as pd
pdidx = pd.IndexSlice
import matplotlib
from matplotlib import pyplot as plt
from matplotlib import colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import seaborn as sns
from numpy import random as rand
from scipy import *
import time as T
import sys
import json
from datetime import datetime
import os

from sklearn.metrics import roc_curve, roc_auc_score, log_loss, accuracy_score, precision_recall_curve,\
                            average_precision_score, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import xgboost as xgb
from xgboost import XGBRegressor
from xgboost import plot_importance
import optuna as opt

# In this notebook, we'll experiment a bit with including other stock information, starting first without time windows.   
# However, first let's try including lag 6 and lag 1

In [4]:
def preprocess_PoC(df, STOCK):
    df = (df
          .set_index(['date_id', 'stock_id', 'seconds_in_bucket'])
          .sort_index(level=['date_id', 'stock_id', 'seconds_in_bucket']))
    
    df = df.drop(['row_id', 'time_id'], axis=1)

    df_stock = df.loc[pd.IndexSlice[:, STOCK, :], :]
    targets_stock = df_stock[['target']]
    df_stock = df_stock.drop(['target'], axis=1)

    return df_stock, targets_stock

In [5]:
STOCK = 168
print(f"RETAINING DATA FROM STOCK {STOCK} INTO x_wide, y_wide")
path_to_data = "../../train.csv"
ALL_DATA = pd.read_csv(path_to_data)
x_wide, y_wide = preprocess_PoC(ALL_DATA,STOCK)
%reset_selective ALL_ASSETS
x_wide = x_wide.droplevel('stock_id')
y_wide = y_wide.droplevel('stock_id')

RETAINING DATA FROM STOCK 168 INTO x_wide, y_wide


Once deleted, variables cannot be recovered. Proceed (y/[n])?   y


In [6]:
def log_and_clean(df: pd.DataFrame, cols):
    df = df.copy()
    df[cols] = np.log(df[cols])
    df[cols] = df[cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    return df
    
RAW_SIZE_COLS = ['bid_size', 'ask_size', 'matched_size', 'imbalance_size', 'bid_ask_lot_spread']
RAW_PRICE_COLS = ['wap', 'relative_wap']
BASE_LOG_TARGETS = RAW_SIZE_COLS + RAW_PRICE_COLS
LAG_LOG_TARGETS = [f"{c}_lag_6" for c in BASE_LOG_TARGETS]

LOG_COLS = BASE_LOG_TARGETS + LAG_LOG_TARGETS + ['wap_frac'] # wap_frac should not get lagged, it already contains a lag

In [8]:
# --- 1. Feature Construction ---
bid_p, ask_p = x_wide['bid_price'], x_wide['ask_price']
bid_s, ask_s = x_wide['bid_size'], x_wide['ask_size']

rel_wap = (bid_p * ask_s - ask_p * bid_s) / (bid_s + ask_s)
rel_wap.name = 'relative_wap'

log_rel_wap = rel_wap.copy()
log_rel_wap.name = 'log_relative_wap'

log_wap = x_wide['wap'].copy()
log_wap.name = 'log_wap'

BAspread = x_wide['bid_size'] - x_wide['ask_size']
BAspread.name = 'bid_ask_lot_spread'

# x_almost = pd.concat([x_wide, rel_wap, target_lag_6, dy_lag_6, BAspread], axis=1)
x_almost = pd.concat([x_wide, rel_wap, log_rel_wap, log_wap, BAspread], axis=1)

# --- 3. Identify features for rolling/lagging ---
# exclude = ['clf_prob', 'seconds_in_bucket', 'target_lag_6', 'DY_lag_6']
exclude = ['clf_prob', 'seconds_in_bucket']
features_to_process = [c for c in x_almost.columns if c not in exclude]

# --- 4. Rolling medians (window=6) ---
roll_df = (
    x_almost.groupby('date_id')[features_to_process]
            .rolling(window=6)
            .median()
            .reset_index(level=0, drop=True)
)
roll_df.columns = [f"{c}_roll6" for c in roll_df.columns]

# --- 5. Lag 6  ---
lagged6 = (
    x_almost.groupby('date_id')[features_to_process]
            .shift(6)
            .add_suffix('_lag_6')
)
# --- Lag 1 ---
lagged1 = (
    x_almost.groupby('date_id')[features_to_process]
            .shift(1)
            .add_suffix('_lag_1')
)
#why doing lag 6 for this in previous models? I should have access to all waps at all times. Changing to regular delta.
Dwap = x_almost['wap'] - lagged1['wap_lag_1']
Dwap.name = 'delta_wap'

wap_frac = x_almost['wap'] / lagged1['wap_lag_1']
wap_frac.name = 'wap_frac'

log_wap_frac = wap_frac.copy()
log_wap_frac.name = 'log_wap_frac'

# --- 6. Final features ---
x_new_feats = pd.concat([x_almost, Dwap, wap_frac, log_wap_frac, roll_df, lagged1, lagged6], axis=1).fillna(0)

# --- Log scale large quantities ---
x_STOCK = log_and_clean(x_new_feats, LOG_COLS)


/home/travis/.local/lib/python3.8/site-packages/pandas/core/internals/blocks.py:329: RuntimeWarning: divide by zero encountered in log
  result = func(self.values, **kwargs)
/home/travis/.local/lib/python3.8/site-packages/pandas/core/internals/blocks.py:329: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)


In [9]:
#these are for stock 168 from some previous optuna run
params = {
    'N_EST_BT' : 22,
    'MAX_DEPTH_BT' : 7,
    'N_EST_RT' : 1850,
    'MAX_DEPTH_RT' : 9,
    'MIN_CHILD_WEIGHT' : 130,
    'HUBER_SLOPE' : 6.190184273096807,
    'BASE_SCORE_MULTIPLIER' : 0.04073606670469099,
    'EARLY_STOPPING' : 150
}


In [13]:
end_clf_train = 200
end_reg_train = 400
Dy_STOCK = y_wide - y_wide.groupby(level='date_id').shift(1)
# --- CLASSIFIER DATA ---
X_clf_train = x_STOCK.loc[pdidx[:end_clf_train, :], :].reset_index().drop(columns=['date_id'])
Y_clf_train = Dy_STOCK.loc[pdidx[:end_clf_train, :], :].values

X_clf_valid = x_STOCK.loc[pdidx[end_clf_train+1:, :], :].reset_index().drop(columns=['date_id'])
Y_clf_valid = Dy_STOCK.loc[pdidx[end_clf_train+1:, :], :].values

train_mask = ~np.isnan(Y_clf_train).flatten()
valid_mask = ~np.isnan(Y_clf_valid).flatten()

# Cstar, err = find_baseline(X_clf_train[train_mask],Y_clf_train[train_mask],maxC=10,gridpts=1000)

ytr_bin = (Y_clf_train.flatten()[train_mask] >= 0).astype(int)
yva_bin = (Y_clf_valid.flatten()[valid_mask] >= 0).astype(int)

# --- FIT CLASSIFIER ---
BT = xgb.XGBClassifier(
    objective="binary:logistic",
    tree_method="hist",
    learning_rate=0.05, # SHOULD THIS ALSO BE GETTING PLUGGED INTO OPTUNA?
    n_estimators=params['N_EST_BT'],
    max_depth=params['MAX_DEPTH_BT'],
    eval_metric=["auc", "logloss"]
)
BT.fit(X_clf_train.iloc[train_mask], ytr_bin, 
       eval_set=[(X_clf_valid.iloc[valid_mask], yva_bin)], 
       verbose=False)

# --- REGRESSOR DATA ---
# baseline = Cstar * x_STOCK['imbalance_buy_sell_flag']
# baseline.name = 'baseline'
# x_with_basline = pd.concat([x_STOCK, baseline], axis=1)

# baseline answer as feature
# X_reg_train = x_with_basline.loc[pdidx[end_clf_train+1:end_reg_train,:,:]].reset_index()
X_reg_train = x_STOCK.loc[pdidx[end_clf_train+1:end_reg_train,:]].reset_index()

# Y_reg_train = DY_wide.loc[pdidx[end_clf_train+1:end_reg_train,T_start:T_end,:]].values
Y_reg_train = y_wide.loc[pdidx[end_clf_train+1:end_reg_train, :], :].squeeze().to_numpy()
# Y_resid_train = Y_reg_train - X_reg_train['baseline']


X_reg_valid = x_STOCK.loc[pdidx[end_reg_train+1:,:]].reset_index()
# Y_reg_valid = DY_wide.loc[pdidx[end_reg_train+1:,T_start:T_end,:]].values
Y_reg_valid = y_wide.loc[pdidx[end_reg_train+1:, :], :].squeeze().to_numpy()
# Y_resid_valid = Y_reg_valid - X_reg_valid['baseline']

#Classifier data as feature as well
# X_reg_train["clf_prob"] = BT.predict_proba(X_reg_train.drop(columns=['baseline',"date_id"]))[:, 1]
# X_reg_valid["clf_prob"] = BT.predict_proba(X_reg_valid.drop(columns=['baseline',"date_id"]))[:, 1]
X_reg_train["clf_prob"] = BT.predict_proba(X_reg_train.drop(columns=["date_id"]))[:, 1]
X_reg_valid["clf_prob"] = BT.predict_proba(X_reg_valid.drop(columns=["date_id"]))[:, 1]
gt = X_reg_train.groupby(["date_id"])["clf_prob"]
gv = X_reg_valid.groupby([ "date_id"])["clf_prob"]
# also lag_6
X_reg_train["clf_prob_lag6"] = gt.shift(6)
X_reg_valid["clf_prob_lag6"] = gv.shift(6)
# also lag_1
X_reg_train["clf_prob_lag1"] = gt.shift(1)
X_reg_valid["clf_prob_lag1"] = gv.shift(1)
# also rolling mean 6 to get a sense of momentum
X_reg_train["clf_prob_rollmean6"] = gt.rolling(6, min_periods=1).mean().droplevel(list(gt.grouper.names))
X_reg_valid["clf_prob_rollmean6"] = gv.rolling(6, min_periods=1).mean().droplevel(list(gv.grouper.names))

X_reg_train = X_reg_train.drop(columns=["date_id"])
X_reg_valid = X_reg_valid.drop(columns=["date_id"])  


reg_train_mask = ~np.isnan(Y_reg_train).flatten()
reg_valid_mask = ~np.isnan(Y_reg_valid).flatten()

# --- FIT REGRESSOR ---
y_reg_tr_clean = Y_reg_train.flatten()[reg_train_mask]
y_reg_vld_clean = Y_reg_valid.flatten()[reg_valid_mask]

# y_resid_train_clean = Y_resid_train.values[reg_train_mask]
# y_resid_valid_clean = Y_resid_valid.values[reg_valid_mask]

# if len(y_reg_tr_clean) == 0 or len(y_reg_vld_clean) == 0:
#     print(f"Skipping bin {T_start}-{T_end}: No Regressor data.")
#     raise opt.exceptions.TrialPruned()

current_base_score = np.mean(y_reg_tr_clean) * params['BASE_SCORE_MULTIPLIER']
RT = xgb.XGBRegressor(
    n_estimators=params['N_EST_RT'],
    learning_rate=0.005,
    max_depth=params['MAX_DEPTH_RT'],
    min_child_weight=params['MIN_CHILD_WEIGHT'],
    objective='reg:pseudohubererror',
    huber_slope=params['HUBER_SLOPE'],
    base_score=current_base_score,
    tree_method='hist',
    early_stopping_rounds=params['EARLY_STOPPING'],
)
## CHANGED TO FITTING RESIDUALS!!!!
RT.fit(X_reg_train.iloc[reg_train_mask], y_reg_tr_clean,# y_resid_train_clean
       eval_set=[(X_reg_valid.iloc[reg_valid_mask], y_reg_vld_clean)], 
       verbose=False)

# --- ACCUMULATE ERROR ---
        ###### BASELINE ADDED HERE BECAUSE WE FIT RESIDUALS
preds = RT.predict(X_reg_valid.iloc[reg_valid_mask])# + X_reg_valid.iloc[reg_valid_mask]["baseline"].values
actuals = y_reg_vld_clean 

total_absolute_error = np.sum(np.abs(preds - actuals))
# total_square_error += np.sum(np.square(preds - actuals))
total_samples = len(actuals)

MAE = total_absolute_error / total_samples
print(MAE)

3.7996640554190773
